v62 

| Pillar | Winner | Reason |
| :--- | :--- | :--- |
| **1. Momentum** | `VIX Filtered Momentum` | Includes the VIX "Off-Switch" logic. |
| **2. Risk-Adjusted** | `Sharpe (ATRP)` | ATR-based volatility adjustment is more robust for ranking than standard StdDev. |
| **3. Quality** | `Consistency (WinRate 5d)` | Low overlap with Sharpe (18%), measures "steadiness" rather than just total return. |
| **4. Mean Reversion**| `Oversold (-RSI)` | Pure anti-momentum play. 0% overlap with Sharpe. |
| **5. Protection** | `Dip Buyer (-dd_21)` | Captures "Recovery Alpha." Only 29% overlap with RSI. |
| **6. Volatility** | `Low Volatility (-ATRP)` | Picks the "Quiet" stocks. 0% overlap with Momentum. |

To recap for your project notes:
1.  **The Root Cause:** We identified an **Object Reference Mutation** bug where multiple metrics were sharing the same `Series` object. Renaming the `.name` attribute for one metric was overwriting the previous ones, leading to the Pandas naming collisions (`_4`, `_10`, etc.) and the resulting `NaN` columns during alignment.
2.  **The Fix:** Adding `.copy()` to the metric calculation loop in `compute_alpha_ensemble` isolated the data and metadata for each feature.
3.  **The Result:** You now have a clean, 22-column `final_cache_df` and a robust suite of verification tools (`PostBuildVerifier`, `DeepDiveDebugger`, and `ForensicExporter`) to ensure your RL agent always receives high-quality, normalized data.

- convert codebase for RL agent  
- headless scorer
- cross-sectional normalization
- discovery composite action
- temporal vectorizer


v61 
- **Verified all metric calculation with Excel** 
- Updated core.analyzer
- Replaced the `Result` pattern with exceptions and flattened the logic
- Refactored the `AlphaEngine` into a streamlined Orchestrator pattern
-  **Strict Date Logic:** No more "Time Travel" bugs.
-  **Dataclass Contracts:** No more "Magic String" typos or blind dictionaries.
-  **Exception Flow:** The `run` method is now a clean, readable story instead of a maze of `if/else` checks.
-  **Performance Workers:** Math is separated from orchestration.
- Ret_1d, explicitly turns division-by-zero results (`inf`) into `NaN`, and replace [np.inf, -np.inf] with np.nan



v60  
- Converted code from notebook to modular system.
- Fixed divide by zero warning from calculate_gain
- Added subtitle to subplots
- Added Volatility Regime plot


v59  
- Removed "nest" of if-statements in **AlphaEngine.run**
- Use **Result Pattern** to handle errors
- Change verify_analyzer_short and verify_analyzer_long gain calculation from simple return to logarithmic return
- Change calculate_gain from simple return to logarithmic return
- Remove bfill from calculate_gain to prevent backfill with future data
- Verify macro_df calculation


In [60]:
# 1. Enable Autoreload
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def add_project_root_to_path():
    """Find notebooks_RLVR and add to sys.path."""
    current = Path.cwd()

    # Search upward for notebooks_RLVR folder
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            print(f"✓ Added to path: {path}")
            return path
        # Also check if notebooks_RLVR exists as child (for running from stocks/)
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            print(f"✓ Added to path: {candidate}")
            return candidate

    raise RuntimeError("Could not find notebooks_RLVR directory")


# Run once at notebook start
add_project_root_to_path()


# 2. Force reload cached modules (run this to refresh code changes)
modules_to_reload = [
    "core.analyzer",
    "core.auditor",
    "core.builder",
    "core.contracts",    
    "core.engine",
    "core.environment",
    "core.features",
    "core.logic",
    "core.paths",
    "core.performance",
    "core.quant",
    "core.result",
    "core.settings",
    "core.utils",
    "strategy.registry",
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]


# 3. Standard imports
import os
import pandas as pd
import numpy as np
# import multiprocessing
import gc

from IPython.display import display


# 4. Fresh imports (these will re-import from disk due to cache clearing above)
# from core.quant import QuantUtils
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
# from core.logic import AlphaLogic, SelectionLogic
from core.logic import AlphaLogic
from core.paths import OUTPUT_DIR
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import METRIC_REGISTRY


# 5. Pandas display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.precision", 4)


# 6. Load data path from .env
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")

# # 7. Instantiate engine (customize DataFrames as needed)
# master_engine_0 = AlphaEngine(
#     df_ohlcv=df_ohlcv,
#     features_df=features_df,
#     macro_df=macro_df,
#     df_close_wide=df_close_wide,
#     df_atrp_wide=df_atrp_wide,
#     df_trp_wide=df_trp_wide,
# )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Added to path: c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR
NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


In [61]:
#### Change path to frozen data ####
DATA_PATH_OHLCV = r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs_20260324.parquet"
DATA_PATH_INDICES = (
    r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices_20260324.parquet"
)
print(f"=== Overwrite DATA_PATHS ===")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")

=== Overwrite DATA_PATHS ===
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices_20260324.parquet
DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs_20260324.parquet


In [62]:
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
print(f"df_indices:\n{df_indices}")

df_indices:
                   Adj Open  Adj High  Adj Low  Adj Close  Volume
Ticker Date                                                      
^AXJO  1992-11-22   1455.00   1455.00  1455.00    1455.00       0
       1992-11-23   1458.40   1458.40  1458.40    1458.40       0
       1992-11-24   1467.90   1467.90  1467.90    1467.90       0
       1992-11-25   1459.00   1459.00  1459.00    1459.00       0
       1992-11-26   1458.90   1458.90  1458.90    1458.90       0
...                     ...       ...      ...        ...     ...
^VIX3M 2026-03-18     25.00     26.57    24.82      26.56       0
       2026-03-19     27.96     28.03    25.19      25.54       0
       2026-03-20     26.07     28.61    25.91      27.43       0
       2026-03-23     25.60     26.59    24.73      26.10       0
       2026-03-24     26.86     27.20    25.79      26.56       0

[144848 rows x 5 columns]


In [63]:
print(f"Takes about 1.5 minutes")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
print(f"df_ohlcv:\n{df_ohlcv}")

Takes about 1.5 minutes
df_ohlcv:
                   Adj Open  Adj High  Adj Low  Adj Close    Volume
Ticker Date                                                        
A      1999-11-18   27.1966   29.8864  23.9091    26.3000  74849975
       1999-11-19   25.6649   25.7023  23.7970    24.1333  18230876
       1999-11-22   24.6936   26.3000  23.9465    26.3000   7871813
       1999-11-23   25.4034   26.0759  23.9091    23.9091   7151083
       1999-11-24   23.9838   25.0672  23.9091    24.5442   5795948
...                     ...       ...      ...        ...       ...
ZWS    2026-03-18   45.0500   45.1200  44.2300    44.3400   1162000
       2026-03-19   43.6700   44.6200  43.4000    44.0400   1077900
       2026-03-20   44.2400   44.3300  43.0600    43.7900   3255800
       2026-03-23   45.0000   45.9600  44.4700    45.1600   1209900
       2026-03-24   45.0100   45.8100  44.6100    45.4700    825400

[9528174 rows x 5 columns]


In [ ]:
print(f"df_ohlcv.info():\n{df_ohlcv.info()}")

print("=" * 20)

print(f"df_indices.info():\n{df_indices.info()}")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 9528174 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-03-24 00:00:00'))
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Adj Open   float64
 1   Adj High   float64
 2   Adj Low    float64
 3   Adj Close  float64
 4   Volume     int64  
dtypes: float64(4), int64(1)
memory usage: 658.5+ MB
df_ohlcv.info():
None
<class 'pandas.core.frame.DataFrame'>
MultiIndex: 144848 entries, ('^AXJO', Timestamp('1992-11-22 00:00:00')) to ('^VIX3M', Timestamp('2026-03-24 00:00:00'))
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Adj Open   144848 non-null  float64
 1   Adj High   144848 non-null  float64
 2   Adj Low    144848 non-null  float64
 3   Adj Close  144848 non-null  float64
 4   Volume     144848 non-null  int64  
dtypes: float64(4), int64(1)
memory usage: 6.6+ MB
df_indices.info():
None


In [64]:
print(f"Takes about 3 minutes to generate_features")

features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

Takes about 3 minutes to generate_features
⚡ Generating Decoupled Features (Benchmark: SPY)...


In [65]:
print(
    "🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)"
)

# 1. Price Matrix
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)

# 2. Volatility Matrices (Unstack and Align)
# Using features_df (the first item from the tuple)
print("   - Unstacking ATRP...")
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)

print("   - Unstacking TRP...")
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# 3. Handle Data Gaps (Sanitize the Wide Matrices)
if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill up to the limit, then fill remaining with the "Disaster Detection" value
df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pre-computation Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)
print("   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.")

🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)
   - Unstacking ATRP...
   - Unstacking TRP...
✅ Pre-computation Complete. Tickers: 1585, Days: 16164
   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.


In [66]:
# _indices = df_indices.index.get_level_values(0).unique().tolist()
# display(_indices)
# df_indices.info()

# # print(f"df_ohlcv.head():\n {df_ohlcv.head()}\n")
# df_ohlcv.info()

In [67]:
# print(f"df_close_wide.info():\n{df_close_wide.info()}")
# print(f"df_atrp_wide.info():\n{df_atrp_wide.info()}")
# print(f"df_trp_wide.info():\n{df_trp_wide.info()}")

In [68]:
# print(f"features_df.info():\n{features_df.info()}\n")
# print(f"features_df.index.names:\n{features_df.index.names}\n")
# print(f"macro_df.info():\n{macro_df.info()}\n")
# print(f"macro_df.index.names:\n{macro_df.index.names}\n")

In [69]:
# # Pre-flight Automated Audit Suite
# checks = [
#     SA.verify_math_integrity(),
#     SA.verify_ranking_integrity(),
#     SA.verify_vol_alignment_integrity(),
#     SA.verify_feature_engineering_integrity(),
# ]

# for check in checks:
#     icon = "✅" if check.ok else "🔥"
#     print(f"{icon} {check.msg}")
#     if not check.ok:
#         print("🛑 Critical Failure. System Halted.")
#         break

# print("=" * 85)
# # Separate verify_marco_engine output from intertwine with other outputs

# checks = [
#     SA.verify_macro_engine(
#         df_ohlcv=df_ohlcv,
#         df_indices=df_indices,
#         original_macro_df=macro_df,
#         settings=GLOBAL_SETTINGS,
#     ),
# ]

# for check in checks:
#     icon = "✅" if check.ok else "🔥"
#     print(f"{icon} {check.msg}")
#     if not check.ok:
#         print("🛑 Critical Failure. System Halted.")
#         break

#### Run ParallelFeatureBuilder

In [ ]:
# # # --- THE MARATHON CONFIG ---

# # Instantiate engine (customize DataFrames as needed)
# master_engine_0 = AlphaEngine(
#     df_ohlcv=df_ohlcv,
#     features_df=features_df,
#     macro_df=macro_df,
#     df_close_wide=df_close_wide,
#     df_atrp_wide=df_atrp_wide,
#     df_trp_wide=df_trp_wide,
# )

CHECKPOINT_DIR = "_alpha_cache_v1_checkpoints"
FINAL_FILE = OUTPUT_DIR / "AlphaCache_Master_2025.parquet"
# I recommend leaving 2 cores for your PC.
# If you have an 8-core CPU, it will use 6 for the Bake.
# WORKER_COUNT = max(1, multiprocessing.cpu_count() - 2)

# ParallelFeatureBuilder.run_marathon(
#     master_engine=master_engine_0,
#     lookbacks=[10, 21],
#     start_date="2025-01-01",
#     checkpoint_dir=CHECKPOINT_DIR,
#     batch_size=50,
#     # num_workers=WORKER_COUNT,  # <--- TAME THE BEAST
#     num_workers=1,  # <--- TAME THE BEAST
#     debug_mode=False,
# )

# ParallelFeatureBuilder.run_marathon(
#     master_engine=master_engine_0,
#     lookbacks=[10, 21],
#     start_date="2025-01-01",
#     checkpoint_dir=CHECKPOINT_DIR,
#     batch_size=1,
#     num_workers=1,  # One worker ensures prints show up in order
#     debug_mode=False,
#     debug_sample_dates=["2025-01-02"],  # <--- This will force it to run ONLY this date
# )

# # 2. THE STITCH (Run this when the progress bar hits 100%)
# final_cache_df = FeatureCubeStitcher.assemble(CHECKPOINT_DIR, FINAL_FILE)
# print(f"final_cache_df:\n{final_cache_df}")

In [74]:
# Load the final_cache_df from the parquet file
final_cache_df = pd.read_parquet(
    # "AlphaCache_Master_2025.parquet",
    FINAL_FILE,
    engine="pyarrow",
)
print(f"Loaded final_cache_df from {FINAL_FILE}")
print(f"Shape: {final_cache_df.shape}")
print(f"final_cache_df:\n{final_cache_df}")

Loaded final_cache_df from C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\AlphaCache_Master_2025.parquet
Shape: (926, 22)
final_cache_df:
                   10d_Price_Gain  10d_Sharpe  10d_Sharpe_ATRP  10d_Sharpe_TRP  10d_Momentum_21d  10d_Info_Ratio_Stdev_Alpha_63d  10d_Consistency_WinRate_5d  10d_Oversold_RSI  10d_Dip_Buyer_Drawdown_dd_21  10d_Low_Volatility_ATRP  10d_VIX_Filtered_Momentum  21d_Price_Gain  21d_Sharpe  21d_Sharpe_ATRP  21d_Sharpe_TRP  21d_Momentum_21d  21d_Info_Ratio_Stdev_Alpha_63d  21d_Consistency_WinRate_5d  21d_Oversold_RSI  21d_Dip_Buyer_Drawdown_dd_21  21d_Low_Volatility_ATRP  21d_VIX_Filtered_Momentum
Date       Ticker                                                                                                                                                                                                                                                                                                                                        

In [87]:
# ============================================================
# ✅ KEEP THESE FUNCTIONS — Delete everything else
# ============================================================


class PostBuildVerifier:
    """
    The Structural "Gatekeeper"
    Stateless Post-build verification suite for Z-score validation.
    Run this AFTER FeatureCubeStitcher.assemble() completes.
    **Why:** This is your most robust tool.
    It verifies that the Z-scores are mathematically correct (Mean ~0, Std ~1)
    across all dates. If this fails, your RL agent will receive garbage data.
    """

    @staticmethod
    def generate_summary_report(df: pd.DataFrame):
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        print("\n" + "=" * 60 + "\nFINAL CACHE SUMMARY\n" + "=" * 60)
        print(
            f"📐 Shape: {df.shape} | 🏷️ Tickers: {df.index.get_level_values('Ticker').nunique()}"
        )

        all_vals = df[numeric_cols].values.flatten()
        all_vals = all_vals[~np.isnan(all_vals)]
        print(
            f"🌍 Global Z: Mean={np.mean(all_vals):.6f}, Std={np.std(all_vals, ddof=1):.6f}"
        )
        print(
            f"📊 NaN%: {100 * np.isnan(df[numeric_cols].values).sum() / df[numeric_cols].size:.2f}%"
        )

    @staticmethod
    def verify_normality(df: pd.DataFrame, sample_size=5, tolerance=1e-10):
        dates = df.index.get_level_values("Date").unique()
        sample_dates = dates[np.linspace(0, len(dates) - 1, sample_size, dtype=int)]
        numeric_cols = df.select_dtypes(include=[np.number]).columns

        print(f"\n🔍 Checking Normality ({len(sample_dates)} dates)...")
        for dt in sample_dates:
            slice_df = df.loc[dt, numeric_cols]
            max_mean = slice_df.mean().abs().max()
            max_std_err = (slice_df.std(ddof=1) - 1.0).abs().max()

            status = "✅" if (max_mean < tolerance and max_std_err < 0.01) else "❌"
            print(
                f"   {dt.date()}: Mean_Err={max_mean:.2e}, Std_Err={max_std_err:.2e} {status}"
            )

    @staticmethod
    def detect_anomalies(df: pd.DataFrame, limit=3.5):
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        extreme = (df[numeric_cols].abs() > limit).sum().sum()
        if extreme > 0:
            print(f"⚠️  Detected {extreme} values outside +/- {limit} sigma.")
        else:
            print(f"✅ No extreme anomalies detected.")


class RLVRParityVerifier:
    """
    The Functional "Logic Check"
    **Why:** It answers the question: *"Does the new cache select the same tickers as my old engine?" * and *
    "Is the reward calculated correctly?"*
    This ensures your training pipeline won't break when you swap the data source.
    """

    @staticmethod
    def verify(
        cache_df: pd.DataFrame,
        df_close_wide: pd.DataFrame,
        engine_output: any,
        lookback: int,
        registry_metric: str,
        rank_start: int = 1,  # New param
        rank_end: int = 10,  # New param
    ):
        # 1. Modular Name Mapping
        clean_metric = AlphaLogic.slugify_columns([registry_metric])[0]
        cache_col = f"{lookback}d_{clean_metric}"
        decision_date = engine_output.decision_date

        # 2. Selection Parity: Slice by Rank
        # We sort descending (highest Z-score first) and take the rank range
        day_slice = cache_df.xs(decision_date, level="Date")[cache_col].dropna()

        # rank_start=1, rank_end=10 -> .iloc[0:10]
        cache_tickers = (
            day_slice.sort_values(ascending=False)
            .iloc[rank_start - 1 : rank_end]
            .index.tolist()
        )

        engine_tickers = engine_output.tickers

        # 3. REWARD LOGIC: REPLICATING QuantUtils.compute_portfolio_stats
        entry_date = engine_output.buy_date
        end_date = engine_output.holding_end_date

        # Slice the window [Entry : End]
        price_window = df_close_wide.loc[entry_date:end_date, cache_tickers]

        # a) Normalize prices to the START of the holding period (Entry Date)
        norm_prices = price_window.div(price_window.iloc[0])

        # b) Equal weights at the start (1/N)
        weights = 1.0 / len(cache_tickers)

        # c) Equity Curve = Sum of (Relative Price * Weight)
        equity_curve = (norm_prices * weights).sum(axis=1)

        # d) Calculate Total Period Log Return
        final_equity = equity_curve.iloc[-1]
        vectorized_log_reward = np.log(final_equity)

        # 4. Comparison
        legacy_reward = engine_output.perf_metrics.get("holding_p_gain", 0.0)
        gain_diff = abs(vectorized_log_reward - legacy_reward)

        print(
            f"--- PARITY REPORT: {decision_date.date()} | {registry_metric} ({lb}d) ---"
        )
        print(f"Rank Range:           {rank_start} to {rank_end}")
        print(
            f"Engine Tickers: {engine_tickers}\n"
            f"Cache Tickers:  {cache_tickers}\n"
            f"Ticker Match:         {'✅ PASS' if set(cache_tickers) == set(engine_tickers) else '❌ FAIL'}"
        )
        if set(cache_tickers) != set(engine_tickers):
            print(f"  - Missing in Cache:  {set(engine_tickers) - set(cache_tickers)}")
            print(f"  - Extra in Cache:    {set(cache_tickers) - set(engine_tickers)}")

        print(f"Log Reward Match:     {'✅ PASS' if gain_diff < 1e-7 else '❌ FAIL'}")
        print(f"  - Vectorized (Log): {vectorized_log_reward:.8f}")
        print(f"  - Legacy (Log):     {legacy_reward:.8f}")
        print(f"  - Delta:            {gain_diff:.2e}")


class DeepDiveDebugger:
    """
    Consolidated debugging tool for reconciling Cache Z-scores against Raw OHLCV.
    """

    @staticmethod
    def audit_date(df_ohlcv, cache_df, date, lookback=10, metric_slug="Price_Gain"):
        """
        Reconciles an entire date's cross-sectional calculation.
        """
        dt = pd.Timestamp(date)
        col = f"{lookback}d_{metric_slug}"

        # 1. Align Dates & Universes
        all_dates = df_ohlcv.index.get_level_values("Date").unique().sort_values()
        if dt not in all_dates:
            dt = all_dates[all_dates.get_indexer([dt], method="nearest")[0]]

        end_idx = all_dates.get_loc(dt)
        start_date = all_dates[max(0, end_idx - lookback)]

        cache_slice = cache_df.xs(dt, level="Date")[col].dropna()

        # 2. Universal Intersection
        idx = pd.IndexSlice
        ohlcv_now = (
            df_ohlcv.loc[idx[:, dt], :].index.get_level_values("Ticker").unique()
        )
        ohlcv_start = (
            df_ohlcv.loc[idx[:, start_date], :]
            .index.get_level_values("Ticker")
            .unique()
        )
        ohlcv_valid = set(ohlcv_now).intersection(ohlcv_start)

        common = set(cache_slice.index).intersection(ohlcv_valid)

        print(f"\n{'='*60}\nAUDIT: {dt.date()} | {col}\n{'='*60}")
        print(
            f"Universe: Cache({len(cache_slice)}) | OHLCV({len(ohlcv_valid)}) | Common({len(common)})"
        )

        # 3. Recalculate Log Returns
        recalc = []
        for t in common:
            p0 = df_ohlcv.loc[idx[t, start_date], "Adj Close"]
            p1 = df_ohlcv.loc[idx[t, dt], "Adj Close"]
            # Handle duplicates if index isn't perfectly clean
            p0 = p0.iloc[0] if isinstance(p0, pd.Series) else p0
            p1 = p1.iloc[0] if isinstance(p1, pd.Series) else p1

            log_ret = np.log(p1 / p0)
            recalc.append(
                {"Ticker": t, "raw_ret": log_ret, "cache_z": cache_slice.loc[t]}
            )

        df = pd.DataFrame(recalc).set_index("Ticker")

        # 4. Cross-sectional Math Validation
        mean_ret = df["raw_ret"].mean()
        std_ret = df["raw_ret"].std(ddof=1)
        df["calc_z"] = (df["raw_ret"] - mean_ret) / std_ret
        df["diff"] = (df["calc_z"] - df["cache_z"]).abs()

        print(f"Stats: Mean={mean_ret:.6f}, Std={std_ret:.6f}")
        print(f"Max Z-Diff: {df['diff'].max():.2e}")
        print(f"Matches (<1e-10): {(df['diff'] < 1e-10).sum()} / {len(df)}")

        return df

    @staticmethod
    def ticker_walk(df_ohlcv, cache_df, ticker, date, lookback=10):
        """
        Detailed price-level walk for a specific ticker.
        """
        dt = pd.Timestamp(date)
        all_dates = df_ohlcv.index.get_level_values("Date").unique().sort_values()
        end_idx = all_dates.get_loc(dt)
        start_date = all_dates[max(0, end_idx - lookback)]

        idx = pd.IndexSlice
        try:
            p0 = df_ohlcv.loc[idx[ticker, start_date], "Adj Close"]
            p1 = df_ohlcv.loc[idx[ticker, dt], "Adj Close"]
            p0 = p0.iloc[0] if isinstance(p0, pd.Series) else p0
            p1 = p1.iloc[0] if isinstance(p1, pd.Series) else p1

            log_ret = np.log(p1 / p0)
            z_score = cache_df.loc[(dt, ticker), f"{lookback}d_Price_Gain"]

            print(f"\n--- TICKER WALK: {ticker} ---")
            print(f"Start ({start_date.date()}): ${p0:.4f}")
            print(f"End   ({dt.date()}): ${p1:.4f}")
            print(f"Log Return: {log_ret:.6f}")
            print(f"Cache Z:    {z_score:.6f}")
        except KeyError:
            print(f"❌ Ticker {ticker} or Date missing in OHLCV/Cache.")


class ForensicExporter:
    """
    Recreates the 'Worker-style' debug CSV for any date.
    Shows RAW values -> STATS -> Z-SCORES in one file.
    """

    @staticmethod
    def export_date_reconciliation(
        master_engine, date, lookbacks, output_dir="reconciliation"
    ):
        dt = pd.Timestamp(date)
        os.makedirs(output_dir, exist_ok=True)

        # 1. Get RAW values from the engine (Un-normalized)
        # Note: We need the engine that has the registry and data access
        raw_ensemble = master_engine.compute_alpha_ensemble(dt, lookbacks)
        if raw_ensemble.empty:
            print(f"❌ No data found for {dt.date()}")
            return

        clean_df = raw_ensemble.dropna()

        # 2. Calculate Stats (Excel Parity: ddof=1)
        means = clean_df.mean()
        stds = clean_df.std(ddof=1)
        counts = clean_df.count()

        # 3. Calculate Z-Scores
        zscores = (clean_df - means) / stds

        # 4. Format for Excel
        # a) RAW block
        raw_block = clean_df.copy()
        raw_block.insert(0, "__Stage__", "1_RAW")

        # b) STATS block
        stats_data = []
        for label, vals in [
            ("__MEAN__", means),
            ("__STD__", stds),
            ("__COUNT__", counts),
        ]:
            row = {col: vals[col] for col in clean_df.columns}
            row["__Stage__"] = f"2_STATS_{label}"
            stats_data.append(pd.Series(row, name=label))
        stats_block = pd.DataFrame(stats_data)

        # c) Z-SCORE block
        z_block = zscores.copy()
        z_block.insert(0, "__Stage__", "3_ZSCORE")

        # 5. Combine and Slugify columns (so headers match final_cache_df)
        final_csv_df = pd.concat([raw_block, stats_block, z_block])
        final_csv_df.columns = AlphaLogic.slugify_columns(final_csv_df.columns.tolist())

        # 6. Save
        fn = f"{output_dir}/forensic_{dt.strftime('%Y%m%d')}.csv"
        final_csv_df.to_csv(fn)

        print(f"\n✅ Forensic file created: {fn}")
        print(f"   Tickers: {len(clean_df)}")
        print(f"   Columns: {len(clean_df.columns)}")
        print(f"👉 Open this in Excel to verify: (Raw - Mean) / Std = Zscore")

        return fn


#

In [88]:
verification_date = "2025-01-02"
rank_start = 1
rank_end = 10
lookback_period = 10
holding_period = 5


# 1. Define the Action (Inputs)
_input = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp(verification_date),
    lookback_period=lookback_period,
    holding_period=holding_period,
    metric="Sharpe (ATRP)",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=rank_start,
    rank_end=rank_end,
    debug=True,
)

# 6. Instantiate engine (customize DataFrames as needed)
master_engine_1 = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

In [89]:
import re

# --- 1. CONFIGURATION & DISCOVERY ---
# Auto-detect lookbacks from column names (e.g., "21d_Sharpe" -> 21)
all_cols = final_cache_df.columns.tolist()
lookbacks = sorted(
    list(
        set(
            [
                int(re.search(r"^(\d+)d_", col).group(1))
                for col in all_cols
                if re.search(r"^\d+d_", col)
            ]
        )
    )
)

print(f"🔍 Detected Lookback Periods in Cache: {lookbacks}")
print(f"📊 Metrics in Registry: {len(METRIC_REGISTRY)}")
print(f"🧪 Expected Total Columns: {len(lookbacks) * len(METRIC_REGISTRY)}")

# --- 2. STRUCTURAL INTEGRITY (Tier 1) ---
PostBuildVerifier.generate_summary_report(final_cache_df)
PostBuildVerifier.verify_normality(final_cache_df, sample_size=5)
PostBuildVerifier.detect_anomalies(final_cache_df, limit=4.0)

# --- 3. LOGIC & REWARD PARITY (Tier 2) ---
# We loop through every metric and every lookback detected
print(f"\n{'='*60}\nRUNNING CROSS-ENGINE PARITY CHECK\n{'='*60}")
unique_dates = final_cache_df.index.get_level_values("Date").unique()

if len(unique_dates) == 0:
    print("❌ ERROR: No dates found in final_cache_df. Cannot run parity check.")
else:
    # Safely pick a date (use the first one available)
    parity_date = unique_dates[0]
    print(f"✅ Running parity check for date: {parity_date.date()}")

    print(f"\n{'='*20} Testing Lookback: {lookback_period}d {'='*20}")
    for metric_name in METRIC_REGISTRY.keys():
        print(f"Testing Metric: {metric_name}")

        # 1. Update the input for the legacy engine
        _input.metric = metric_name
        _input.decision_date = parity_date
        _input.lookback = lookback_period  # Ensure engine knows which lookback to use

        # 2. Run the legacy engine
        engine_output = master_engine_1.run(_input)

        # 3. Verify against the new cache
        RLVRParityVerifier.verify(
            cache_df=final_cache_df,
            df_close_wide=df_close_wide,
            engine_output=engine_output,
            lookback=lookback_period,
            registry_metric=_input.metric,
            rank_start=_input.rank_start,  # Pass these through
            rank_end=_input.rank_end,
        )


# --- 4. DEEP DIVE (Tier 3: Run manually if Tier 2 fails) ---
# Example: If '10d_Price_Gain' fails for Ticker 'A'
# DeepDiveDebugger.audit_date(df_ohlcv, final_cache_df, parity_date, lookback=10)
# DeepDiveDebugger.ticker_walk(df_ohlcv, final_cache_df, "A", parity_date, lookback=10)

🔍 Detected Lookback Periods in Cache: [10, 21]
📊 Metrics in Registry: 11
🧪 Expected Total Columns: 22

FINAL CACHE SUMMARY
📐 Shape: (926, 22) | 🏷️ Tickers: 926
🌍 Global Z: Mean=-0.000000, Std=0.999484
📊 NaN%: 0.00%

🔍 Checking Normality (5 dates)...
   2025-01-02: Mean_Err=3.11e-16, Std_Err=1.44e-15 ✅
   2025-01-02: Mean_Err=3.11e-16, Std_Err=1.44e-15 ✅
   2025-01-02: Mean_Err=3.11e-16, Std_Err=1.44e-15 ✅
   2025-01-02: Mean_Err=3.11e-16, Std_Err=1.44e-15 ✅
   2025-01-02: Mean_Err=3.11e-16, Std_Err=1.44e-15 ✅
⚠️  Detected 95 values outside +/- 4.0 sigma.

RUNNING CROSS-ENGINE PARITY CHECK
✅ Running parity check for date: 2025-01-02

==================== Testing Lookback: 10d ====================
Testing Metric: Price Gain
DEBUG: 926 stocks passed filters on 2025-01-02
--- PARITY REPORT: 2025-01-02 | Price Gain (21d) ---
Rank Range:           1 to 10
Engine Tickers: ['DRI', 'AR', 'NVDL', 'APA', 'FTAI', 'NXT', 'DLTR', 'HUM', 'EQNR', 'VST']
Cache Tickers:  ['DRI', 'AR', 'NVDL', 'APA', 'FT

In [ ]:
DeepDiveDebugger.audit_date(
    df_ohlcv, final_cache_df, parity_date, lookback=lookback_period
)
print(f"\n{'=' * 20}")
DeepDiveDebugger.ticker_walk(
    df_ohlcv, final_cache_df, "A", parity_date, lookback=lookback_period
)


AUDIT: 2025-01-02 | 10d_Price_Gain
Universe: Cache(926) | OHLCV(1555) | Common(926)
Stats: Mean=-0.031414, Std=0.045883
Max Z-Diff: 3.55e-15
Matches (<1e-10): 926 / 926


--- TICKER WALK: A ---
Start (2024-12-17): $135.1210
End   (2025-01-02): $132.3640
Log Return: -0.020615
Cache Z:    0.235365


In [79]:
# Recreate the check for Jan 2nd
ForensicExporter.export_date_reconciliation(
    master_engine=master_engine_1,
    date=verification_date,
    lookbacks=[lookback_period],  # or [21, 63, 189] depending on your bake
)


✅ Forensic file created: reconciliation/forensic_20250102.csv
   Tickers: 926
   Columns: 11
👉 Open this in Excel to verify: (Raw - Mean) / Std = Zscore


'reconciliation/forensic_20250102.csv'

In [ ]:
# 1. Define the Action (Inputs)
_input = EngineInput(
    mode="Ranking",
    # decision_date=pd.Timestamp(verification_date),
    decision_date=pd.Timestamp("2026-03-20"),
    lookback_period=lookback_period,
    holding_period=holding_period,
    metric="Sharpe (ATRP)",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=rank_start,
    rank_end=rank_end,
    debug=True,
)

# 2. Run the Headless Engine
metrics_df = run_headless_simulation(master_engine_1, _input)

# 3. Verify Result
print("--- HEADLESS METRICS REPORT ---")
display(metrics_df)

# TEST: Gain in Holding Period (The Reward for RL)
reward = metrics_df.loc["Group Gain", "Holding"]
print(f"\nTarget Reward Signal: {reward:.6f}")
#

🚨 Engine Error: ❌ Decision Date too late. Latest valid: 2026-03-16
--- HEADLESS METRICS REPORT ---


""


KeyError: 'Holding'

In [81]:
#############################
#############################
#############################
#############################
#############################
#############################

In [134]:
_decision_date = "2026-03-16"
_lookback_period = 63
_holding_period = 5
_rank_start = 1
_rank_end = 100
_debug = False
_benchmark = GLOBAL_SETTINGS["benchmark_ticker"]

In [135]:
# 1. Create a "Template" Input to check the date
_input_check = EngineInput(
    mode="Ranking",
    decision_date=pd.Timestamp(_decision_date),
    lookback_period=_lookback_period,
    holding_period=_holding_period,
    metric="Check",  # Metric name doesn't matter for timeline validation
    benchmark_ticker=_benchmark,
    rank_start=_rank_start,
    rank_end=_rank_end,
    debug=_debug,
)

try:
    # 2. Call the validation method directly
    # This will raise a ValueError if the date is too late/early
    master_engine_1._validate_timeline(_input_check)

    # 3. If we get here, the date is VALID. Proceed to loop.
    metric_results = {}
    for metric_name in METRIC_REGISTRY.keys():
        _input_check.metric = metric_name
        result = master_engine_1.run(_input_check)
        metric_results[metric_name] = result.tickers
        print(f"Processed {metric_name}: Found {len(result.tickers)} tickers.")

except ValueError as e:
    # 4. Catch the specific error from _validate_timeline
    print(f"🚨 Engine Error: {e}")
    print("--- HEADLESS METRICS REPORT ---")

except Exception as e:
    print(f"An unexpected error occurred: {e}")

DEBUG: 942 stocks passed filters on 2026-03-16
Processed Price Gain: Found 100 tickers.
DEBUG: 942 stocks passed filters on 2026-03-16
Processed Sharpe: Found 100 tickers.
DEBUG: 942 stocks passed filters on 2026-03-16
Processed Sharpe (ATRP): Found 100 tickers.
DEBUG: 942 stocks passed filters on 2026-03-16
Processed Sharpe (TRP): Found 100 tickers.
DEBUG: 942 stocks passed filters on 2026-03-16
Processed Momentum (21d): Found 100 tickers.
DEBUG: 942 stocks passed filters on 2026-03-16
Processed Info Ratio (Stdev_Alpha, 63d): Found 100 tickers.
DEBUG: 942 stocks passed filters on 2026-03-16
Processed Consistency (WinRate 5d): Found 100 tickers.
DEBUG: 942 stocks passed filters on 2026-03-16
Processed Oversold (-RSI): Found 100 tickers.
DEBUG: 942 stocks passed filters on 2026-03-16
Processed Dip Buyer (Drawdown -dd_21): Found 100 tickers.
DEBUG: 942 stocks passed filters on 2026-03-16
Processed Low Volatility (-ATRP): Found 100 tickers.
DEBUG: 942 stocks passed filters on 2026-03-16
P

In [139]:
from collections import Counter, defaultdict
import textwrap

# --- THE MISSING BRIDGE ---
# 1. Flatten all ticker lists from metric_results into one long list
all_tickers_combined = []
for tickers in metric_results.values():
    all_tickers_combined.extend(tickers)

# 2. Count how many times each ticker appears across all metrics
ticker_counts = Counter(all_tickers_combined)
# ---------------------------

# 3. Group tickers by their frequency (Conviction Score)
conviction_groups = defaultdict(list)
threshold = 5
for ticker, count in ticker_counts.items():
    if count >= threshold:
        conviction_groups[count].append(ticker)

# 4. Print the Formatted Report
print("=" * 60)
print(f"       📊 MULTI-METRIC CONSENSUS REPORT ({_decision_date})")
print("=" * 60)
print(
    f"Analysis across {len(metric_results)} metrics | Threshold: {threshold}+ Metrics"
)
print("-" * 60)

# Sort groups from highest conviction to lowest
for score in sorted(conviction_groups.keys(), reverse=True):
    tickers = sorted(conviction_groups[score])
    ticker_str = ", ".join(tickers)

    # Visual indicators based on conviction
    if score >= 8:
        prefix = "💎"
    elif score >= 6:
        prefix = "⭐"
    else:
        prefix = "✅"

    print(f"\n{prefix} CONVICTION SCORE: {score} Metrics")
    print(
        textwrap.fill(ticker_str, width=60, initial_indent="  ", subsequent_indent="  ")
    )

print("\n" + "=" * 60)
total_consensus = sum(len(t) for t in conviction_groups.values())
print(f"TOTAL UNIQUE CONSENSUS TICKERS: {total_consensus}")
print("=" * 60)

       📊 MULTI-METRIC CONSENSUS REPORT (2026-03-16)
Analysis across 11 metrics | Threshold: 5+ Metrics
------------------------------------------------------------

💎 CONVICTION SCORE: 8 Metrics
  APA, CVX, EOG, ERIC, NOK, SHEL, SNDK, USO, VLO

⭐ CONVICTION SCORE: 7 Metrics
  CF, CNQ, COP, CVE, DOW, LNG, LYB, MRNA, OVV, OXY, PBR, PR,
  SU, TPL, TRGP

⭐ CONVICTION SCORE: 6 Metrics
  BP, ED, EIX, EQIX, EWY, LIN, MU, NTR, WMB, XLE, XOM

✅ CONVICTION SCORE: 5 Metrics
  AEP, ASX, ATI, BALL, BG, CASY, DTE, DUK, EPD, EVRG, FDX,
  FTI, HON, KEYS, KMI, LHX, LMT, MINT, MSI, MTZ, NOC, PCG,
  SCHD, SHV, TDY, USFR, VRT, VZ

TOTAL UNIQUE CONSENSUS TICKERS: 63


In [141]:
# Updated styling function to handle high-contrast visibility
def highlight_high_overlap(val):
    # If the value is 1.0 (diagonal), mute it
    if val >= 0.999:
        return "background-color: #333333; color: #777777; border: 1px solid #444"

    # High Correlation (> 80%) - Bright Red Background, White Text
    if val > 0.80:
        return "background-color: #d9534f; color: white; font-weight: bold; border: 1px solid #444"

    # Medium Correlation (> 60%) - Orange Background, White Text
    if val > 0.60:
        return "background-color: #f0ad4e; color: white; border: 1px solid #444"

    # Low Correlation - Dark Grey Background, Light Text (Ensures visibility)
    return "background-color: #222222; color: #cccccc; border: 1px solid #333"


print("--- METRIC SIMILARITY MATRIX (Jaccard Index) ---")

# 1. Use .map instead of .applymap to fix the FutureWarning
# 2. Add .set_table_styles to ensure the headers are also visible
styled_df = overlap_df.style.map(highlight_high_overlap).format("{:.2%}")

styled_df.set_table_styles(
    [
        {
            "selector": "th",
            "props": [
                ("background-color", "#111"),
                ("color", "#fff"),
                ("font-weight", "bold"),
            ],
        },
        {"selector": "td", "props": [("padding", "10px")]},
    ]
)

display(styled_df)

--- METRIC SIMILARITY MATRIX (Jaccard Index) ---


,Price Gain,Sharpe,Sharpe (ATRP),Sharpe (TRP),Momentum (21d),"Info Ratio (Stdev_Alpha, 63d)",Consistency (WinRate 5d),Oversold (-RSI),Dip Buyer (Drawdown -dd_21),Low Volatility (-ATRP),VIX Filtered Momentum
Price Gain,100.00%,33.33%,40.85%,38.89%,24.22%,38.89%,19.76%,0.00%,2.56%,0.00%,24.22%
Sharpe,33.33%,100.00%,73.91%,80.18%,13.64%,68.07%,20.48%,0.00%,0.00%,6.38%,13.64%
Sharpe (ATRP),40.85%,73.91%,100.00%,88.68%,16.28%,61.29%,18.34%,0.00%,0.50%,5.82%,16.28%
Sharpe (TRP),38.89%,80.18%,88.68%,100.00%,16.28%,62.60%,18.34%,0.00%,0.00%,6.38%,16.28%
Momentum (21d),24.22%,13.64%,16.28%,16.28%,100.00%,14.94%,16.28%,0.00%,0.00%,0.00%,100.00%
"Info Ratio (Stdev_Alpha, 63d)",38.89%,68.07%,61.29%,62.60%,14.94%,100.00%,18.34%,0.00%,0.00%,2.56%,14.94%
Consistency (WinRate 5d),19.76%,20.48%,18.34%,18.34%,16.28%,18.34%,100.00%,0.50%,1.01%,2.04%,16.28%
Oversold (-RSI),0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.50%,100.00%,29.03%,1.01%,0.00%
Dip Buyer (Drawdown -dd_21),2.56%,0.00%,0.50%,0.00%,0.00%,0.00%,1.01%,29.03%,100.00%,0.00%,0.00%
Low Volatility (-ATRP),0.00%,6.38%,5.82%,6.38%,0.00%,2.56%,2.04%,1.01%,0.00%,100.00%,0.00%


In [ ]:
# 1. Define the "Best of Breed" subset based on our analysis
proposed_metrics = [
    "Price Gain",
    "Sharpe (ATRP)",  # Risk-Adjusted Trend ==
    "Consistency (WinRate 5d)",  # Quality/Steadiness ===
    "Oversold (-RSI)",  # Mean Reversion ===
    "Dip Buyer (Drawdown -dd_21)",  # Recovery Alpha ===
    "Low Volatility (-ATRP)",  # Volatility Factor
    "VIX Filtered Momentum",  # Pure Trend with Safety Switch
]

# 2. Filter the existing results and calculate the new matrix
n_new = len(proposed_metrics)
pruned_similarity_matrix = np.zeros((n_new, n_new))

for i in range(n_new):
    for j in range(n_new):
        set_i = set(metric_results[proposed_metrics[i]])
        set_j = set(metric_results[proposed_metrics[j]])

        intersection = len(set_i.intersection(set_j))
        union = len(set_i.union(set_j))

        pruned_similarity_matrix[i, j] = intersection / union if union > 0 else 0

# 3. Create the DataFrame
pruned_overlap_df = pd.DataFrame(
    pruned_similarity_matrix, index=proposed_metrics, columns=proposed_metrics
)

# 4. Display the "Lean" Matrix
print("--- PRUNED METRIC SIMILARITY MATRIX (Orthogonal Factors) ---")
display(pruned_overlap_df.style.map(highlight_high_overlap).format("{:.2%}"))

--- PRUNED METRIC SIMILARITY MATRIX (Orthogonal Factors) ---


,Price Gain,Sharpe (ATRP),Consistency (WinRate 5d),Oversold (-RSI),Dip Buyer (Drawdown -dd_21),Low Volatility (-ATRP),VIX Filtered Momentum
Price Gain,100.00%,40.85%,19.76%,0.00%,2.56%,0.00%,24.22%
Sharpe (ATRP),40.85%,100.00%,18.34%,0.00%,0.50%,5.82%,16.28%
Consistency (WinRate 5d),19.76%,18.34%,100.00%,0.50%,1.01%,2.04%,16.28%
Oversold (-RSI),0.00%,0.00%,0.50%,100.00%,29.03%,1.01%,0.00%
Dip Buyer (Drawdown -dd_21),2.56%,0.50%,1.01%,29.03%,100.00%,0.00%,0.00%
Low Volatility (-ATRP),0.00%,5.82%,2.04%,1.01%,0.00%,100.00%,0.00%
VIX Filtered Momentum,24.22%,16.28%,16.28%,0.00%,0.00%,0.00%,100.00%
